In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy.optimize import curve_fit  # Necessário para ajuste ponderado com erros
import glob
import os

In [ ]:
def processar_v0(arquivo, label_cor):
    df = pd.read_csv(arquivo, sep=";", decimal=",", names=["V", "I"], skiprows=1)

    # Cálculo da Segunda Derivada (d2I/dV2)
    d1 = np.gradient(df["I"], df["V"])
    d2 = np.gradient(d1, df["V"])
    df["d2I"] = d2

    # Fit Linear na região de ruído
    mask_ruido = (df["V"] >= -9) & (df["V"] <= -3)
    res = linregress(df.loc[mask_ruido, "V"], df.loc[mask_ruido, "d2I"])

    # Cálculo da banda de confiança de 68% (1 sigma) [cite: 814]
    std_err = df.loc[mask_ruido, "d2I"].std()
    df["fit_y"] = res.intercept + res.slope * df["V"]
    df["upper_68"] = df["fit_y"] + std_err

    # 4. Identificação do V0 (Primeiro ponto que foge da banda) [cite: 816]
    # Procuramos a partir de -3V para a direita (sentido positivo)
    df_busca = df[df["V"] > -3].copy()
    v0_idx = (df_busca["d2I"] > df_busca["upper_68"]).idxmax()
    v0 = df.loc[v0_idx, "V"]

    return df, v0, res, std_err


def ordenar_arquivos(arquivos):
    """Ordena arquivos por ordem do espectro"""
    ordem = {
        "sem": 0,
        "UV": 1,
        "uv": 1,
        "vio": 2,
        "azul": 3,
        "verde": 4,
        "ver": 5,
        "amar": 6,
    }

    def chave_ordenacao(arquivo):
        nome = os.path.basename(arquivo).lower()
        for cor, posicao in ordem.items():
            if nome.startswith(cor.lower()):
                # Extrair intensidade (número após hífen)
                if "-" in nome:
                    try:
                        intensidade = int(nome.split("-")[1].split(".")[0])
                        return (posicao, intensidade)
                    except:
                        return (posicao, 0)
                return (posicao, 0)
        return (999, 0)

    return sorted(arquivos, key=chave_ordenacao)

In [ ]:
# Frequências e Comprimentos de Onda (Ångströms) - conforme "Resultados Esperados"
# Baseado na lâmpada de Mercúrio
frequencias = {
    "UV": 8.22e14,  # ~3650 A
    "uv": 8.22e14,
    "vio": 7.41e14,  # ~4046 A
    "azul": 6.88e14,  # ~4358 A
    "verde": 5.49e14,  # ~5461 A
    "ver": 4.88e14,  # ~6149 A (Vermelho?)
    "amar": 5.19e14,  # ~5779 A
}

# Comprimentos de onda aproximados para tabela (em nm) para cálculo de incerteza da frequencia
lambdas_nm = {
    "UV": 365.0,
    "uv": 365.0,
    "vio": 404.6,
    "azul": 435.8,
    "verde": 546.1,
    "amar": 577.9,
    "ver": 614.9,
}

In [ ]:
# --- PROCESSAMENTO DE TODOS OS ARQUIVOS CSV ---
pasta = r"C:\Users\gabri\Documents\POLI_Docs\7. Lab Fisica\attachments"
arquivos_csv = glob.glob(os.path.join(pasta, "*.csv"))
arquivos_csv = ordenar_arquivos(arquivos_csv)

In [ ]:
resultados = []

for arquivo in arquivos_csv:
    nome_arquivo = os.path.basename(arquivo)
    label = nome_arquivo.replace(".csv", "")

    try:
        df_plot, v0_calc, fit, sigma = processar_v0(arquivo, label)

        # Identificar cor base
        cor_base = None
        for cor in frequencias.keys():
            if label.lower().startswith(cor.lower()):
                cor_base = cor
                break

        resultados.append(
            {
                "arquivo": nome_arquivo,
                "label": label,
                "cor": cor_base,
                "v0": v0_calc,
                "v0_abs": abs(v0_calc),
                "frequencia": frequencias.get(cor_base, None),
                "df": df_plot,
                "fit": fit,
                "sigma": sigma,
            }
        )
    except Exception as e:
        print(f"Erro ao processar {nome_arquivo}: {e}")

In [ ]:
# --- NOVA LÓGICA DE PLOTAGEM AGRUPADA (Estilo Stacked) ---

# 1. Agrupar resultados por 'Cor' base
grupos_cor = {}
for res in resultados:
    c = res["cor"]
    if c is None:
        c = "Outros"
    if c not in grupos_cor:
        grupos_cor[c] = []
    grupos_cor[c].append(res)

# 2. Gerar uma figura para cada Cor contendo subplots das intensidades
for cor, lista_resultados in grupos_cor.items():
    if not lista_resultados:
        continue

    # Ordenar por nome do arquivo para tentar manter ordem de intensidade (opcional)
    lista_resultados.sort(key=lambda x: x["arquivo"])

    n_plots = len(lista_resultados)

    # Configurar figura com subplots compartilhando eixo X e sem espaço vertical (hspace=0)
    fig, axes = plt.subplots(
        n_plots, 1, figsize=(10, 3 * n_plots), sharex=True, gridspec_kw={"hspace": 0}
    )

    # Garantir que axes seja iterável mesmo se for apenas 1 plot
    if n_plots == 1:
        axes = [axes]

    print(f"Gerando gráfico empilhado para cor: {cor} ({n_plots} intensidades)...")

    for i, (ax, res) in enumerate(zip(axes, lista_resultados)):
        df_p = res["df"]
        v0 = res["v0"]
        sig = res["sigma"]
        lbl = res["label"]

        # Gráfico da 2ª Derivada (Azul escuro)
        ax.plot(
            df_p["V"],
            df_p["d2I"],
            color="navy",
            linewidth=1.5,
            label="2ª Derivada" if i == 0 else "",
        )

        # Ajuste Linear de Ruído (Preto)
        ax.plot(
            df_p["V"],
            df_p["fit_y"],
            color="black",
            linewidth=1.5,
            linestyle="-",
            label="Fit Ruído" if i == 0 else "",
        )

        # Banda de Confiança (Cinza)
        ax.fill_between(
            df_p["V"],
            df_p["fit_y"] - sig,
            df_p["fit_y"] + sig,
            color="gray",
            alpha=0.4,
            label="Confiança 68%" if i == 0 else "",
        )

        # Linha Vertical do V0 (Vermelho)
        ax.axvline(
            v0,
            color="red",
            linestyle="-",
            linewidth=2,
            label=f"$V_0$" if i == 0 else "",
        )

        # Texto com valor de V0 no topo da linha
        ylim_max = df_p["d2I"].max()
        ax.text(
            v0,
            ylim_max,
            f"{v0:.2f} V",
            color="red",
            fontweight="bold",
            ha="center",
            va="bottom",
            fontsize=11,
            backgroundcolor="white",
        )

        # Legenda do nome do arquivo dentro do plot (canto esquerdo)
        ax.text(
            0.02,
            0.85,
            f"Arq: {lbl}",
            transform=ax.transAxes,
            fontsize=10,
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
        )

        # Configurações de grade e limites
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.set_xlim(
            -10, 10
        )  # Ajuste conforme seus dados (-10 a 1V parece ser o range da imagem)

        # Remover ticks do eixo X para os gráficos de cima
        if i < n_plots - 1:
            ax.tick_params(labelbottom=False)

        # Labels apenas no último gráfico e no gráfico central
        if i == n_plots - 1:
            ax.set_xlabel("Tensão Aplicada (V)", fontsize=12)

        if i == n_plots // 2:
            ax.set_ylabel("$d^2I/dV^2$ (u.a.)", fontsize=12)

    # Título Geral
    fig.suptitle(
        f"Determinação de $V_0$ - Espectro: {cor}",
        fontsize=16,
        fontweight="bold",
        y=0.90,
    )  # y ajusta altura do titulo

    # Legenda única no topo (usando o primeiro ax)
    lines, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        lines,
        labels,
        loc="upper right",
        bbox_to_anchor=(0.95, 0.89),
        fontsize=10,
        framealpha=1,
    )

    plt.savefig(f"v0_stacked_{cor}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

print(f"\n✓ Gráficos empilhados gerados com sucesso!")

In [ ]:
# ==============================================================================
# ANÁLISE ESTATÍSTICA - MÉTODO DOS SLIDES
# ==============================================================================

# 1. Agrupar resultados por Cor para Estatística
dados_por_cor = {}
for res in resultados:
    if res["cor"] and res["frequencia"]:
        cor = res["cor"]
        if cor not in dados_por_cor:
            dados_por_cor[cor] = {"v0_list": [], "freq": res["frequencia"]}
        dados_por_cor[cor]["v0_list"].append(res["v0_abs"])

# Listas para o ajuste final
freqs_val = []
v0_means = []
v0_errs_total = []  # sigma_v0 (total)
freq_errs = []  # sigma_f

# Parâmetros de Incerteza Instrumentais/Estimados
sigma_inst = 0.01  # Incerteza instrumental do voltímetro (V)
sigma_lambda_nm = 5.0  # Largura de banda do filtro/monocromador (nm) - Estimativa

tabela_final = []

for cor, dados in dados_por_cor.items():
    vals = np.array(dados["v0_list"])
    n = len(vals)
    freq = dados["freq"]

    # --- Cálculos Estatísticos (Slide: Método de Análise - Estatística) ---

    # 1. Média (V0_barra)
    v0_medio = np.mean(vals)

    # 2. Desvio Padrão da Amostra (sigma_v0 na fórmula do slide)
    # Se n=1, o desvio padrão é indefinido (ou zero para fins de plot), usamos só instrumental
    if n > 1:
        std_sample = np.std(vals, ddof=1)
    else:
        std_sample = 0.0

    # 3. Desvio Padrão Médio (sigma_m = sigma_sample / sqrt(n))
    sigma_m = std_sample / np.sqrt(n)

    # 4. Incerteza Total (sigma_t = sqrt(sigma_m^2 + sigma_inst^2))
    sigma_t = np.sqrt(sigma_m**2 + sigma_inst**2)

    # 5. Incerteza da Frequência (Propagação)
    # sigma_f = (c / lambda^2) * sigma_lambda
    lam_nm = lambdas_nm.get(cor, (2.998e8 / freq) * 1e9)  # Pega do dict ou calcula
    lam_m = lam_nm * 1e-9
    sig_lam_m = sigma_lambda_nm * 1e-9
    c_luz = 2.99792458e8

    sigma_f = (c_luz / (lam_m**2)) * sig_lam_m

    # Armazenar dados
    freqs_val.append(freq)
    v0_means.append(v0_medio)
    v0_errs_total.append(sigma_t)
    freq_errs.append(sigma_f)

    tabela_final.append(
        {
            "Lambda (A)": f"{lam_nm*10:.0f}",
            "Freq (Hz)": freq,
            "Sigma_f": sigma_f,
            "V0 Medio (V)": v0_medio,
            "Sigma_V0": sigma_t,
            "N": n,
        }
    )

# Converter para arrays numpy para cálculos
x_data = np.array(freqs_val)
y_data = np.array(v0_means)
x_err = np.array(freq_errs)
y_err = np.array(v0_errs_total)

# Exibir Tabela estilo "Resultados Esperados"
df_resumo_final = pd.DataFrame(tabela_final).sort_values("Freq (Hz)", ascending=False)
print("\n" + "=" * 30 + " TABELA FINAL DE RESULTADOS " + "=" * 30)
print(
    df_resumo_final.to_string(
        index=False,
        formatters={
            "Freq (Hz)": "{:.2e}".format,
            "Sigma_f": "{:.1e}".format,
            "V0 Medio (V)": "{:.2f}".format,
            "Sigma_V0": "{:.2f}".format,
        },
    )
)
print("=" * 86)

# --- AJUSTE LINEAR PONDERADO (Weighted Least Squares) ---
# Usamos curve_fit considerando o sigma_V0 (y_err)


def func_reta(f, h_over_e, phi_over_e):
    return h_over_e * f - phi_over_e


# Ajuste: absolute_sigma=True é crucial para que a matriz de covariância reflita magnitudes reais
popt, pcov = curve_fit(func_reta, x_data, y_data, sigma=y_err, absolute_sigma=True)

slope = popt[0]  # h/e
intercept = -popt[1]  # -Phi (o curve fit retorna o parametro)
slope_err = np.sqrt(pcov[0, 0])
intercept_err = np.sqrt(pcov[1, 1])

# Cálculo das Constantes
e_carga = 1.60217663e-19
h_calculado = slope * e_carga
h_erro = slope_err * e_carga
phi_eV = abs(popt[1])
phi_erro = intercept_err

h_tabelado = 6.62607015e-34
erro_perc = abs((h_calculado - h_tabelado) / h_tabelado) * 100

# --- GRÁFICO FINAL ---
plt.figure(figsize=(10, 7))

# 1. Banda de Confiança do Ajuste (68% - 1 sigma)
# Propagação de erro da reta: sigma_y^2 = (df/da)^2 * var_a + (df/db)^2 * var_b + 2*cov_ab...
# Modelo: y = a*x - b
x_fit = np.linspace(min(x_data) * 0.9, max(x_data) * 1.1, 100)
y_fit = func_reta(x_fit, *popt)

# Variância da predição (Intervalo de confiança da reta média)
var_y_fit = (
    (x_fit**2) * pcov[0, 0] + pcov[1, 1] - 2 * x_fit * pcov[0, 1]
)  # Atenção ao sinal da covariancia dependendo da definição
sigma_y_fit = np.sqrt(var_y_fit)

plt.plot(x_fit, y_fit, color="red", linewidth=2, label="Ajuste Linear Ponderado")
plt.fill_between(
    x_fit,
    y_fit - sigma_y_fit,
    y_fit + sigma_y_fit,
    color="red",
    alpha=0.2,
    label="Banda de Confiança 68%",
)

# 2. Dados Experimentais com Barras de Erro em X e Y
plt.errorbar(
    x_data,
    y_data,
    xerr=x_err,
    yerr=y_err,
    fmt="o",
    color="black",
    ecolor="black",
    capsize=5,
    elinewidth=1.5,
    label="Dados Experimentais ($V_0 \pm \sigma_t$)",
)

# 3. Formatação
plt.title(
    "Potencial de Corte $V_0$ vs Frequência (Análise Estatística)",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Frequência da Radiação ($Hz$)", fontsize=12)
plt.ylabel("Potencial de Corte $V_0$ ($V$)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11, loc="upper left")

# 4. Caixa de Texto com Resultados
results_text = (
    f"Ajuste Ponderado:\n"
    f"$V_0 = ({slope:.2e})f - ({phi_eV:.2f})$\n\n"
    f"$h_{{calc}} = ({h_calculado/1e-34:.2f} \pm {h_erro/1e-34:.2f}) \\times 10^{{-34}} J\\cdot s$\n"  # type: ignore
    f"$h_{{ref}}  = {h_tabelado:.2e} J\\cdot s$\n"
    f"Erro: {erro_perc:.2f}%\n\n"
    f"Função Trabalho $\phi$: ({phi_eV:.2f} $\pm$ {phi_erro:.2f}) eV"  # pyright: ignore[reportInvalidStringEscapeSequence]
)
props = dict(boxstyle="round", facecolor="wheat", alpha=0.5)
plt.text(
    0.95,
    0.05,
    results_text,
    transform=plt.gca().transAxes,
    fontsize=10,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=props,
)

plt.tight_layout()
plt.savefig("v0_planck_final_weighted.png", dpi=300)
plt.show()

print("\n" + "=" * 80)
print(" RESULTADOS FINAIS DA DETERMINAÇÃO DE PLANCK")
print("=" * 80)
print(f"Inclinação (h/e): {slope:.3e} +/- {slope_err:.3e} V/Hz")
print(f"Intercepto (phi): {phi_eV:.3f} +/- {phi_erro:.3f} V")
print(f"Constante h:      {h_calculado:.3e} J.s")
print(f"Incerteza h:      {h_erro:.3e} J.s")
print("=" * 80)